In [2]:
# Imports
import pandas as pd
import numpy as np
import random
import os
import sys

# Config
sys.path.append("..")
from config import *

# Data Loading
dirty_df = pd.read_csv("../DATA/RAW/transactions_dirty_raw.csv")
merchants_df = pd.read_csv("../DATA/RAW/merchants_raw.csv")
user_device_mapping = pd.read_csv("../DATA/RAW/user_device_mapping_raw.csv")
users_df = pd.read_csv("../DATA/RAW/users_raw.csv")

print("Dirty dataset loaded successfully.")
print("Shape:", dirty_df.shape)

Dirty dataset loaded successfully.
Shape: (200300, 10)


In [9]:
# dirty_df.shape
dirty_df.isnull().sum()

transaction_id           0
timestamp                0
user_id                  0
device_id              301
merchant_id            502
transaction_type         0
amount                   0
payment_status         199
suspicion_flag           0
suspicion_reason    189147
dtype: int64

In [10]:
quality_report_before = pd.DataFrame({
    "Missing": dirty_df.isnull().sum(),
    "Data_Type": dirty_df.dtypes.astype(str),
    "Unique_Values": dirty_df.nunique()
})

quality_report_before

,Missing,Data_Type,Unique_Values
transaction_id,0,str,200000
timestamp,0,str,198162
user_id,0,str,10000
device_id,301,str,10597
merchant_id,502,str,500
transaction_type,0,str,3
amount,0,int64,22783
payment_status,199,str,6
suspicion_flag,0,bool,2
suspicion_reason,189147,str,3


In [11]:
dirty_df.duplicated().sum()

np.int64(300)

In [12]:
dirty_df = dirty_df.drop_duplicates().reset_index(drop=True)

In [13]:
# dirty_df.shape
dirty_df.duplicated().sum()

np.int64(0)

In [14]:
dirty_df["merchant_id"].isnull().sum()

np.int64(500)

In [15]:
missing_merchant = dirty_df["merchant_id"].isnull()

dirty_df.loc[missing_merchant, "merchant_id"] = np.random.choice(
    merchants_df["merchant_id"],
    missing_merchant.sum()
)

In [16]:
dirty_df["merchant_id"].isnull().sum()

np.int64(0)

In [17]:
user_device_dict = (
    user_device_mapping
    .groupby("user_id")["device_id"]
    .apply(list)
    .to_dict()
)

In [18]:
device_missing = dirty_df["device_id"].isnull()

dirty_df.loc[device_missing, "device_id"] = (
    dirty_df.loc[device_missing, "user_id"]
    .apply(lambda x: random.choice(user_device_dict[x]))
)

In [19]:
dirty_df["device_id"].isnull().sum()

np.int64(0)

In [20]:
dirty_df["payment_status"] = (
    dirty_df["payment_status"]
    .fillna("Pending")
)

In [21]:
dirty_df["payment_status"].isnull().sum()

np.int64(0)

In [22]:
dirty_df["payment_status"].value_counts(dropna=False)

payment_status
Success    189216
Failed       8402
Pending      2182
Faild          73
Succes         65
Pendng         62
Name: count, dtype: int64

In [23]:
dirty_df["payment_status"] = (
    dirty_df["payment_status"]
    .replace({
        "Succes":"Success",
        "Faild":"Failed",
        "Pendng":"Pending"
    })
)

In [24]:
dirty_df["payment_status"].value_counts()

payment_status
Success    189281
Failed       8475
Pending      2244
Name: count, dtype: int64

In [25]:
# dirty_df.shape

# dirty_df.duplicated().sum()

# dirty_df["merchant_id"].isnull().sum()

# dirty_df["device_id"].isnull().sum()

# dirty_df["payment_status"].isnull().sum()

# dirty_df["payment_status"].value_counts()

In [26]:
# dirty_df.shape

# dirty_df.duplicated().sum()
dirty_df["merchant_id"].isnull().sum()

np.int64(0)

In [27]:
dirty_df.to_csv(
    "../DATA/CLEANED/transactions_cleaned_step1.csv",
    index=False
)

In [28]:
pd.read_csv("../DATA/CLEANED/transactions_cleaned_step1.csv").shape

(200000, 10)

In [29]:
users_df.columns

Index(['user_id', 'account_age_days', 'customer_segment', 'state', 'city'], dtype='str')

In [30]:
dirty_df = dirty_df.merge(

    users_df[["user_id","state","city"]],

    on="user_id",

    how="left"

)

In [31]:
# dirty_df.shape

# dirty_df[["user_id","state","city"]].head()

# dirty_df["state"].isnull().sum()

dirty_df["city"].isnull().sum()

np.int64(0)

In [32]:
space_idx = np.random.choice(
    dirty_df.index,
    500,
    replace=False
)

dirty_df.loc[
    space_idx,
    "city"
] = dirty_df.loc[
    space_idx,
    "city"
].apply(lambda x: f"  {x}  ")

In [33]:
dirty_df.loc[
    space_idx,
    "city"
].head()

58536        Hyderabad  
28573       Coimbatore  
85399         Durgapur  
113361         Dhanbad  
124652       New Delhi  
Name: city, dtype: str

In [34]:
case_idx = np.random.choice(
    dirty_df.index,
    400,
    replace=False
)

dirty_df.loc[
    case_idx,
    "city"
] = dirty_df.loc[
    case_idx,
    "city"
].apply(
    lambda x: random.choice([
        x.lower(),
        x.upper()
    ])
)

In [35]:
hidden_idx = np.random.choice(
    dirty_df.index,
    200,
    replace=False
)

dirty_df.loc[
    hidden_idx,
    "city"
] = dirty_df.loc[
    hidden_idx,
    "city"
].apply(
    lambda x: x.replace(" ", chr(160))
)

In [36]:
dirty_df["city"] = (
    dirty_df["city"]
    .str.strip()
)

In [37]:
dirty_df["city"] = (
    dirty_df["city"]
    .str.replace(chr(160)," ",regex=False)
)

In [38]:
dirty_df["city"] = (
    dirty_df["city"]
    .str.title()
)

In [39]:
dirty_df["state"] = (
    dirty_df["state"]
    .str.strip()
    .str.title()
)

In [40]:
dirty_df["city"].sample(10)

156988       Mysuru
43037      Durgapur
82693      Durgapur
49937       Kolkata
81979       Kolkata
161531        Patna
64872       Lucknow
76789        Mysuru
2594      New Delhi
58638        Jaipur
Name: city, dtype: str

In [41]:
dirty_df[
    dirty_df["city"].str.isupper()
]

,transaction_id,timestamp,user_id,device_id,merchant_id,transaction_type,amount,payment_status,suspicion_flag,suspicion_reason,state,city


In [51]:
dirty_df[
    dirty_df["city"].str.endswith(" ")
]

,transaction_id,timestamp,user_id,device_id,merchant_id,transaction_type,amount,payment_status,suspicion_flag,suspicion_reason,state,city


In [43]:
dirty_df.to_excel(
    "../EXCEL/raw_excel_copy.xlsx",
    index=False
)

In [44]:
dirty_df.to_csv(
    "../DATA/CLEANED/transactions_cleaned_step2.csv",
    index=False
)

In [45]:
pd.read_csv(
    "../DATA/CLEANED/transactions_cleaned_step2.csv"
).shape

(200000, 12)

In [56]:
dirty_df.shape

# dirty_df["state"].isnull().sum()

dirty_df["city"].isnull().sum()

dirty_df["city"].sample(10)

pd.read_csv("../DATA/CLEANED/transactions_cleaned_step2.csv").shape

(200000, 12)

In [57]:
(dirty_df["amount"] < 0).sum()

np.int64(100)

In [58]:
dirty_df["amount_valid"] = dirty_df["amount"] > 0

In [59]:
dirty_df["amount_valid"].value_counts()

amount_valid
True     199900
False       100
Name: count, dtype: int64

In [60]:
dirty_df["timestamp_valid"] = (
    (pd.to_datetime(dirty_df["timestamp"]) >= START_DATE)
    &
    (pd.to_datetime(dirty_df["timestamp"]) <= END_DATE)
)

In [61]:
dirty_df["timestamp_valid"].value_counts()

timestamp_valid
True     198887
False      1113
Name: count, dtype: int64

In [62]:
valid_status = [
    "Success",
    "Failed",
    "Pending"
]

In [63]:
dirty_df["status_valid"] = (
    dirty_df["payment_status"]
    .isin(valid_status)
)

In [64]:
dirty_df["status_valid"].value_counts()

status_valid
True    200000
Name: count, dtype: int64

In [65]:
valid_location_map = (
    users_df
    .groupby("state")["city"]
    .apply(set)
    .to_dict()
)

In [66]:
def validate_location(row):

    return row["city"] in valid_location_map[row["state"]]

In [67]:
dirty_df["location_valid"] = (
    dirty_df.apply(validate_location, axis=1)
)

In [68]:
dirty_df["location_valid"].value_counts()

location_valid
True    200000
Name: count, dtype: int64

In [69]:
dirty_df["quality_score"] = 100

In [70]:
dirty_df.loc[
    ~dirty_df["amount_valid"],
    "quality_score"
] -= 30

In [71]:
dirty_df.loc[
    ~dirty_df["timestamp_valid"],
    "quality_score"
] -= 25

In [72]:
dirty_df.loc[
    ~dirty_df["status_valid"],
    "quality_score"
] -= 20

In [73]:
dirty_df.loc[
    ~dirty_df["location_valid"],
    "quality_score"
] -= 25

In [74]:
dirty_df["quality_score"].describe()

count    200000.000000
mean         99.845875
std           1.975953
min          70.000000
25%         100.000000
50%         100.000000
75%         100.000000
max         100.000000
Name: quality_score, dtype: float64

In [75]:
quality_summary = pd.DataFrame({

    "Metric":[
        "Total Rows",
        "Negative Amount",
        "Invalid Timestamp",
        "Invalid Status",
        "Invalid Location",
        "Average Quality Score"
    ],

    "Value":[

        len(dirty_df),

        (~dirty_df["amount_valid"]).sum(),

        (~dirty_df["timestamp_valid"]).sum(),

        (~dirty_df["status_valid"]).sum(),

        (~dirty_df["location_valid"]).sum(),

        round(dirty_df["quality_score"].mean(),2)

    ]
})

quality_summary

,Metric,Value
0,Total Rows,200000.00
1,Negative Amount,100.00
2,Invalid Timestamp,1113.00
3,Invalid Status,0.00
4,Invalid Location,0.00
5,Average Quality Score,99.85


In [83]:
# dirty_df.shape

# dirty_df["amount_valid"].value_counts()

# dirty_df["timestamp_valid"].value_counts()

# dirty_df["status_valid"].value_counts()

# dirty_df["location_valid"].value_counts()

# dirty_df["quality_score"].describe()

quality_summary

# pd.read_csv("../DATA/CLEANED/transactions_cleaned_final.csv").shape

,Metric,Value
0,Total Rows,200000.00
1,Negative Amount,100.00
2,Invalid Timestamp,1113.00
3,Invalid Status,0.00
4,Invalid Location,0.00
5,Average Quality Score,99.85


In [84]:
END_DATE_FULL = END_DATE + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

In [85]:
dirty_df["timestamp_valid"] = (
    (pd.to_datetime(dirty_df["timestamp"]) >= START_DATE)
    &
    (pd.to_datetime(dirty_df["timestamp"]) <= END_DATE_FULL)
)

In [86]:
dirty_df["timestamp_valid"].value_counts()

timestamp_valid
True    200000
Name: count, dtype: int64

In [87]:
dirty_df["quality_score"] = 100

In [88]:
dirty_df.loc[~dirty_df["amount_valid"], "quality_score"] -= 30
dirty_df.loc[~dirty_df["timestamp_valid"], "quality_score"] -= 25
dirty_df.loc[~dirty_df["status_valid"], "quality_score"] -= 20
dirty_df.loc[~dirty_df["location_valid"], "quality_score"] -= 25

In [89]:
dirty_df["quality_score"].describe()

count    200000.000000
mean         99.985000
std           0.670654
min          70.000000
25%         100.000000
50%         100.000000
75%         100.000000
max         100.000000
Name: quality_score, dtype: float64

In [90]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Negative Amount",
        "Invalid Timestamp",
        "Invalid Status",
        "Invalid Location",
        "Average Quality Score"
    ],
    "Value": [
        len(dirty_df),
        (~dirty_df["amount_valid"]).sum(),
        (~dirty_df["timestamp_valid"]).sum(),
        (~dirty_df["status_valid"]).sum(),
        (~dirty_df["location_valid"]).sum(),
        round(dirty_df["quality_score"].mean(), 2)
    ]
})

quality_summary

,Metric,Value
0,Total Rows,200000.00
1,Negative Amount,100.00
2,Invalid Timestamp,0.00
3,Invalid Status,0.00
4,Invalid Location,0.00
5,Average Quality Score,99.98


In [91]:
dirty_df.to_csv(
    "../DATA/CLEANED/transactions_cleaned_final.csv",
    index=False
)

quality_summary.to_csv(
    "../DATA/CLEANED/data_quality_summary.csv",
    index=False
)

In [92]:
pd.read_csv("../DATA/CLEANED/transactions_cleaned_final.csv").shape

(200000, 17)